In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utils

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")



In [0]:
catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

base_path=f"s3://sportsbar-aws-dp/{data_source}/*.csv"


#Bronze Processing

In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(base_path)\
    .withColumn("read_timestamp",F.current_timestamp())\
    .select("*","_metadata.file_name","_metadata.file_size")
display(df.limit(10))

In [0]:
df.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed","true")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

#Silver Processing

In [0]:
df_bronze=spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_silver=df.dropDuplicates(['customer_id'])

In [0]:
display(
    df_silver.filter(F.col("customer_name")!=F.trim(F.col("customer_name")))
)

In [0]:
df_silver=df_silver.withColumn("customer_name",F.trim(F.col("customer_name")))
df_silver.show(10)

In [0]:
df_silver.select(F.col("city")).distinct().show()

In [0]:
city_mapping={"Hyderbad":"Hyderabad","Bengaluruu":"Bengaluru","Bengalore":"Bengaluru","NewDheli":"New Delhi","NewDelhee":"New Delhi"}
allowed=["Bengaluru","Hyderabad","New Delhi"]
df_silver=df_silver.replace(city_mapping,subset=["city"]).withColumn("city",F.when(F.col("city").isNull(),None)\
    .when(F.col("city").isin(allowed),F.col("city")).otherwise(None))



In [0]:
df_silver.where(F.col("customer_name").isin(df_silver.where(F.col("city").isNull()).select(F.col("customer_name")))).show(10)

In [0]:
customer_city_fix={789403:"New Delhi",789420:"Bengaluru",789521:"Hyderabad",789603:"Hyderabad"}
df_fix=spark.createDataFrame([(k,v) for k,v in customer_city_fix.items()],schema=["customer_id","fixed_city"])
display(df_fix)

In [0]:
df_silver=df_silver.join(df_fix,df_silver.customer_id==df_fix.customer_id,how="left")\
    .drop(df_fix.customer_id)\
    .withColumn("city",F.coalesce(F.col("city"),F.col("fixed_city"))).drop("fixed_city")
df_silver.show(10)



In [0]:

df_silver=df_silver.withColumn("customer_id",F.col("customer_id").cast("string")).withColumn("customer",F.concat_ws("-",F.col("customer_name"),F.lit("_"),F.col("city")))\
    .withColumn("market",F.lit("India")).withColumn("platform",F.lit("sports bar")).withColumn("channel",F.lit("acquisition"))

In [0]:
df_silver.printSchema()

In [0]:
df_silver.write.mode("overwrite").format("delta").option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true").saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

#Gold Processing


In [0]:
df_gold=df_silver.select("customer_id","customer","market","city","platform","channel")
df_gold.show(10)

In [0]:
df_gold.write.mode("overwrite")\
    .format("delta").option("delta.enableChangeDataFeed","true").saveAsTable(f"{catalog}.{gold_schema}.sb_{data_source}")

In [0]:
spark.sql("select * from fmcg.gold.dim_customers").show()

In [0]:
delta_table=DeltaTable.forName(spark,"fmcg.gold.dim_customers")
df_child_customers=spark.table("fmcg.gold.sb_customers")\
    .select(F.col("customer_id").alias("customer_code")\
        ,"customer","market","platform","channel")

In [0]:
delta_table.alias("t").merge(source=df_child_customers.alias("s"),condition="t.customer_code=s.customer_code")\
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
%sql
select * from fmcg.gold.dim_customers limit 10